# Imports

In [1]:
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

# import some common libraries
import numpy as np
import os, json, cv2, random, importlib
import matplotlib.pyplot as plt

# import some common detectron2 utilities
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog
import random
import cv2
import matplotlib.pyplot as plt
from detectron2.utils.visualizer import Visualizer, ColorMode
from detectron2.config import get_cfg

# import our functions 
os.chdir("../../../util/preprocessing")
import tumor_dataset

importlib.reload(tumor_dataset)
from tumor_dataset import Dataset

In [2]:
d = Dataset(data_path="../../data/raw_data/useable_data")
d.convert_binary_to_coco()
d.register_instances(rgb=True)

Init Dataset
Created 411 annotations for images in folder: /projects/PUCHALLA/LLP2024/tumor-segmentation/data/processed_data/rgb/train/masks
Created 88 annotations for images in folder: /projects/PUCHALLA/LLP2024/tumor-segmentation/data/processed_data/rgb/val/masks
Created 89 annotations for images in folder: /projects/PUCHALLA/LLP2024/tumor-segmentation/data/processed_data/rgb/test/masks
Created 411 annotations for images in folder: /projects/PUCHALLA/LLP2024/tumor-segmentation/data/processed_data/depth/train/masks
Created 88 annotations for images in folder: /projects/PUCHALLA/LLP2024/tumor-segmentation/data/processed_data/depth/val/masks
Created 89 annotations for images in folder: /projects/PUCHALLA/LLP2024/tumor-segmentation/data/processed_data/depth/test/masks
Created 411 annotations for images in folder: /projects/PUCHALLA/LLP2024/tumor-segmentation/data/processed_data/rgd/train/masks
Created 88 annotations for images in folder: /projects/PUCHALLA/LLP2024/tumor-segmentation/data

In [3]:
train_metadata = MetadataCatalog.get("my_dataset_train")
train_dataset_dicts = DatasetCatalog.get("my_dataset_train")

val_metadata = MetadataCatalog.get("my_dataset_val")
val_dataset_dicts = DatasetCatalog.get("my_dataset_val")

test_metadata = MetadataCatalog.get("my_dataset_test")
test_dataset_dicts = DatasetCatalog.get("my_dataset_test")

[08/09 19:12:55 d2.data.datasets.coco]: Loaded 411 images in COCO format from /projects/PUCHALLA/LLP2024/tumor-segmentation/data/processed_data/rgb/train/images/train.json
[08/09 19:12:55 d2.data.datasets.coco]: Loaded 88 images in COCO format from /projects/PUCHALLA/LLP2024/tumor-segmentation/data/processed_data/rgb/val/images/val.json
[08/09 19:12:55 d2.data.datasets.coco]: Loaded 89 images in COCO format from /projects/PUCHALLA/LLP2024/tumor-segmentation/data/processed_data/rgb/test/images/test.json


# Create Model

In [4]:
# create cfg for model below

iterations = 5000
model_name = "testing-new-LR-4-5000"

cfg = get_cfg()
cfg.MODELNAME = model_name
cfg.OUTPUT_DIR = f"../../../../models/rgb-testing/{cfg.MODELNAME}"
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
cfg.DATASETS.TRAIN = ("my_dataset_train", "my_dataset_val")
cfg.DATASETS.TEST = ("my_dataset_test",)
cfg.DATALOADER.NUM_WORKERS = 1
cfg.DATALOADER.FILTER_EMPTY_ANNOTATIONS = False # this is for our "no tumor" examples 
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")  # Let training initialize from model zoo
cfg.SOLVER.IMS_PER_BATCH = 2  # This is the real "batch size" commonly known to deep learning people
cfg.SOLVER.MAX_ITER = iterations   
cfg.SOLVER.STEPS = []        
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 512   # The "RoIHead batch size". 128 is faster, and good enough for this toy dataset (default: 512)
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1  

# # ROI Head Configuration (Accuracy-focused)
# cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 512
# cfg.MODEL.ROI_HEADS.POSITIVE_FRACTION = 0.5  # More positive samples
# cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.3   # Lower detection threshold
# cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST = 0.3     # Lower NMS for medical

cfg.SOLVER.BASE_LR = 0.0005   # Conservative LR for medical data
cfg.SOLVER.STEPS = [3000, 4000]  # Later LR reduction
cfg.SOLVER.GAMMA = 0.5        # Gentler LR decay
cfg.SOLVER.WARMUP_ITERS = 800
cfg.SOLVER.WARMUP_FACTOR = 0.1

In [5]:
import torch
# additonal cfg info
cfg.MODEL.MASK_ON = True

cfg.MODEL.WEIGHTS = os.path.join("/projects/PUCHALLA/LLP2024/tumor-segmentation/models/rgb-testing/AP-bug-fix-2-5000/model_final.pth")  # path to the model we just trained
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5   # set a custom testing threshold
cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
predictor = DefaultPredictor(cfg)

[08/09 19:12:56 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /projects/PUCHALLA/LLP2024/tumor-segmentation/models/rgb-testing/AP-bug-fix-2-5000/model_final.pth ...


# Evaluate

## Test with default Evaluator

In [11]:
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.modeling import build_model
from detectron2.data import build_detection_test_loader
from detectron2.data.dataset_mapper import DatasetMapper

In [12]:
model = build_model(cfg) 
model.eval()

GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res2): Sequential(
        (0): BottleneckBlock

In [14]:
evaluator = COCOEvaluator("my_dataset_test", cfg, distributed=False, output_dir="./output_eval")
loader = build_detection_test_loader(cfg, "my_dataset_test", mapper=DatasetMapper(cfg, is_train=False))

[08/09 18:25:14 d2.evaluation.coco_evaluation]: Fast COCO eval is not built. Falling back to official COCO eval.
WARNING [08/09 18:25:14 d2.evaluation.coco_evaluation]: COCO Evaluator instantiated using config, this is deprecated behavior. Please pass in explicit arguments instead.
[08/09 18:25:14 d2.data.dataset_mapper]: [DatasetMapper] Augmentations used in inference: [ResizeShortestEdge(short_edge_length=(800, 800), max_size=1333, sample_style='choice')]
[08/09 18:25:14 d2.data.datasets.coco]: Loaded 89 images in COCO format from /projects/PUCHALLA/LLP2024/tumor-segmentation/data/processed_data/rgb/test/images/test.json
[08/09 18:25:14 d2.data.build]: Distribution of instances among all 1 categories:
|  category  | #instances   |
|:----------:|:-------------|
|   Tumor    | 89           |
|            |              |
[08/09 18:25:14 d2.data.common]: Serializing the dataset using: <class 'detectron2.data.common._TorchSerializedList'>
[08/09 18:25:14 d2.data.common]: Serializing 89 e

In [15]:
results = inference_on_dataset(model, loader, evaluator)
print(results['segm'])  # contains AP, AP50, AP75 ...

[08/09 18:25:17 d2.evaluation.evaluator]: Start inference on 89 batches


/home/am0532/.conda/envs/tumor-env/lib/python3.12/site-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4314.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


[08/09 18:25:19 d2.evaluation.evaluator]: Inference done 11/89. Dataloading: 0.0008 s/iter. Inference: 0.0967 s/iter. Eval: 0.0027 s/iter. Total: 0.1002 s/iter. ETA=0:00:07
[08/09 18:25:24 d2.evaluation.evaluator]: Inference done 60/89. Dataloading: 0.0009 s/iter. Inference: 0.0982 s/iter. Eval: 0.0028 s/iter. Total: 0.1019 s/iter. ETA=0:00:02
[08/09 18:25:27 d2.evaluation.evaluator]: Total inference time: 0:00:08.620709 (0.102627 s / iter per device, on 1 devices)
[08/09 18:25:27 d2.evaluation.evaluator]: Total inference pure compute time: 0:00:08 (0.097848 s / iter per device, on 1 devices)
[08/09 18:25:27 d2.evaluation.coco_evaluation]: Preparing results for COCO format ...
[08/09 18:25:27 d2.evaluation.coco_evaluation]: Saving results to ./output_eval/coco_instances_results.json
[08/09 18:25:27 d2.evaluation.coco_evaluation]: Evaluating predictions with official COCO API...
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation